In [1]:
%load_ext autoreload
%autoreload 2

### Install SDG
```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

In [6]:
# Third Party
from datasets import load_dataset

# First Party
from sdg_hub import Flow, FlowRegistry
import os

In [7]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()  

In [ ]:
def set_model_config(flow_object, model_provider):
    # Set model provider
    if model_provider == 'hosted_vllm':    
        vllm_model= getattr(os.getenv('VLLM_MODEL'), 'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct')
        vllm_api_base= getattr(os.getenv('VLLM_API_BASE'), 'localhost:8000/v1')
        vllm_api_key= getattr(os.getenv('VLLM_API_KEY'), 'EMPTY')
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
        )
    elif model_provider == 'openai':
        openai_api_key = os.getenv('OPENAI_API_KEY')
        openai_model = getattr(os.getenv('OPENAI_MODEL'), 'openai/gpt-5')
        openai_api_base = getattr(os.getenv('OPENAI_API_BASE'), 'http://openai:8000/v1')
        flow_object.set_model_config(
            model=openai_model,
            api_base=openai_api_base,
            api_key=openai_api_key,
        )
    elif model_provider == 'ollama':
        ollama_model = getattr(os.getenv('OLLAMA_MODEL'), 'ollama/gemma3')
        ollama_api_base = getattr(os.getenv('OLLAMA_API_BASE'), 'http://localhost:11434')
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
        )
    return flow_object 

In [ ]:
# Creating seed data
# For this example we will use QuALITY Benchmark https://arxiv.org/pdf/2112.08608
quality_corpus = load_dataset("zitongyang/entigraph-quality-corpus", split='train').remove_columns(['entity', 'entigraph']).rename_columns({'raw': 'document', 'uid': 'document_outline'})

# For knowledge tuning we need seed examples (teacher model will use this template to generate more data) to generate synthetic data. We will only use question based on document as seed example. For answers we will let model answer based on document.
seed_examples = {
    "icl_document": (
      "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting a decline in tourism and fishing industries. Residents have organized weekly clean-up initiatives, but the scale of the problem overwhelms their efforts.",
      "Technologists at the local university have developed an AI-powered buoy system to combat this. The buoys, equipped with solar panels and filtration technology, can identify and absorb oil spills while collecting microplastics. Data from the buoys is shared publicly, raising awareness and pressuring corporations to adopt sustainable practices. Though costly, the project has sparked hope for revitalizing the ecosystem and economy."
    ),
    "icl_query_1": "How does the technological solution address the economic *and* environmental challenges highlighted in the document?",
    
    "icl_query_2": "What implicit values or priorities do the community’s actions (clean-up initiatives) and the technologists’ project reflect, and how do these align or contrast?",
    
    "icl_query_3": "Imagine the buoy project succeeds. What unintended consequences might arise from its impact, considering document's themes?",
    "domain": "articles/essays"
}

# Add seed examples to the our corpus
quality_corpus = quality_corpus.map(lambda x: seed_examples)
DOC_UIDS = [
    ' Defining Decay Down by David Plotz',
    ' Fight Clubbed by David Plotz',
    ' I, Antichrist? by Jeffrey Goldberg',
    " It's Time To Keelhaul U-Haul! by Jeffrey Goldberg",
    " My Father's Estate by Ben Stein",
    '"Phone Me in Central Park" by McConnell, James V.',
    'A Coffin for Jacob by Ludwig, Edward W.',
    'A Fall of Glass by Lee, Stanley R.',
    'A Filbert Is a Nut by Raphael, Rick',
    'A Gift from Earth by Banister, Manly',
    'A Gleeb for Earth by Schafhauser, Charles',
    'A Good Year for the Roses? by David Edelstein',
    'A Pail of Air by Leiber, Fritz',
    'A Planet Named Joe by Hunter, Evan',
    "AI: what's the worst that could happen? by Harry Armstrong",
    'Accidental Death by Baily, Peter',
    'All Day September by Kuykendall, Roger',
    'Ambition by Bade, William L.',
    'And Then the Town Took Off by Wilson, Richard',
    'Atom Mystery [Young Atom Detective] by Coombs, Charles Ira',
    'Beach Scene by King, Marshall',
    'Big Ancestor by Wallace, F. L. (Floyd L.)',
    'Birds of a Feather by Silverberg, Robert',
    'Bodyguard by Gold, H. L. (Horace Leonard)'
]
quality_corpus = quality_corpus.filter(lambda x: x['document_outline'] in DOC_UIDS)
quality_corpus.to_json("seed_data_val.jsonl", orient='records', lines=True)

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 194.84ba/s]


634354

### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

#### Discover the available generation flows

In [ ]:
# Load the seed data
number_of_samples = 1
seed_data_dir = f"sdg_demo_output/"
ds = load_dataset('json', data_files=f'seed_data.jsonl', split='train')
ds = ds.shuffle(seed=42).select(range(number_of_samples))

In [8]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[19:26:55] INFO     Discovered 4 flows                                                              ]8;id=111585;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=131526;file:///workspace/home/lab/abhi/sdg_hub_pr_178/src/sdg_hub/core/flow/registry.py#110\110]8;;\

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━┓
┃ Name                                                                            ┃ Author   ┃ Tags     ┃ Descri… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━┩
│ Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning │ SDG Hub  │ questio… │ A       │
│                                                                                 │ Contrib… │ knowled… │ compre… │
│                                                                                 │          │ qa-pair… │ flow    │
│                                                                                 │          │ documen… │ that    │
│                                                                                 │          │ educati… │ genera… │
│                                                                                 │          │          │ high-q… │
│                                                                                 │          │          │ questi… │
│                                                                                 │          │          │ pairs   │
│                                                                                 │          │          │ from    │
│                                                                                 │          │          │ input   │
│                                                                                 │          │          │ docume… │
│                                                                                 │          │          │ using   │
│                                                                                 │          │          │ multip… │
│                                                                                 │          │          │ LLM     │
│                                                                                 │          │          │ blocks  │
│                                                                                 │          │          │ for     │
│                                                                                 │          │          │ questi… │
│                                                                                 │          │          │ genera… │
│                                                                                 │          │          │ answer  │
│                                                                                 │          │          │ synthe… │
│                                                                                 │          │          │ and     │
│                                                                                 │          │          │ quality │
│                                                                                 │          │          │ evalua… │
│ Detailed Summary Knowledge Tuning Dataset Generation Flow                       │ SDG Hub  │ knowled… │ Genera… │
│                                                                                 │ Contrib… │ documen… │ traini… │
│                                                                                 │          │ questio… │ datase… │
│                                                                                 │          │ knowled… │ for     │
│                                                                                 │          │ qa-pair… │ knowle… │
│                                                                                 │          │ documen… │ tuning  │
│                                                                                 │          │ educati… │ by      │
│                                                                                 │          │ multi-s… │ creati… │
│                                                       

Available flows: ['Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning', 'Extractive Summary Knowledge Tuning Dataset Generation Flow', 'Key Facts Knowledge Tuning Dataset Generation Flow', 'Detailed Summary Knowledge Tuning Dataset Generation Flow']
QA flows: ['Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning', 'Extractive Summary Knowledge Tuning Dataset Generation Flow', 'Key Facts Knowledge Tuning Dataset Generation Flow', 'Detailed Summary Knowledge Tuning Dataset Generation Flow']


In [ ]:
# We will use below mapping of flow names to their respective summarization flows
block_name_map = {
        'Detailed Summary Knowledge Tuning Dataset Generation Flow': 'gen_detailed_summary',
        'Key Facts Knowledge Tuning Dataset Generation Flow': 'gen_atomic_facts',
        'Extractive Summary Knowledge Tuning Dataset Generation Flow': 'gen_extractive_summary',
    }

In [ ]:
# Generate data for extractive summary
flow_name = "Extractive Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Generate data for extractive summary
number_of_summaries = 50
extractive_summary_generated_data = flow.generate(ds, runtime_params={
        block_name_map[flow_name]: {
            'n': number_of_summaries
        },
    })

In [ ]:
# Generate similar data for Detailed Summary
flow_name = "Detailed Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)


# Generate data for detailed summary
number_of_summaries = 50
detailed_summary_generated_data = flow.generate(ds, runtime_params={
        block_name_map[flow_name]: {
            'n': number_of_summaries
        },
    })

In [ ]:
# Generate similar data for key facts 
flow_name = "Key Facts Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)


# Generate data for key facts summary
number_of_summaries = 50
key_facts_generated_data = flow.generate(ds, runtime_params={
        block_name_map[flow_name]: {
            'n': number_of_summaries
        },
    })